### Day 1 Assignment: The Lakehouse, Delta Lake & **Notebooks**

### Basic Tasks 

#### 1. Difference between a warehouse, a lake, and a lakehouse. 
A Warehouse is like a structured Database that can store huge amount of data. It can store highly structured data and semi-structured like json but mainly it is used for heavy SQL based workloads like BI, dashboards for faster execution. And it has ACID support. 

A data lake is a vast storage which stores raw, unfiltered, unstructured and semi-structured data. It can store csv, parquet, avro etc. fileformats. It can be used for building ML models and just storing unorganized data in a cost-effective manner. It does not have a strong ACID support.

A data lakehouse is a combination of data lake and data warehouse. It can store all structured, semi-structured, and unstructured data types. It can be used for Data Engineering, Analytics, AI/ML etc. It has ACID support which makes the transactions safe and keep history as versions.

#### 2. Creating a Delta table 

In [0]:
create table if not exists dev.demo.products 
 (product_id int,
  name string,
  category string,
  price float)

#### 3. Running  separate INSERT/UPDATE statements

In [0]:
insert into dev.demo.products
values 
    (1, 'iPhone', 'electronics', 1000),
    (2, 'Macbook', 'electronics', 2000),
    (3, 'Airpods', 'electronics', 100),
    (4, 'T-Shirt', 'clothing', 20),
    (5, 'Jeans', 'clothing', 50),
    (6, 'Shirt', 'clothing', 30),
    (7, 'Pants', 'clothing', 40),
    (8, 'Socks', 'clothing', 5),
    (9, 'Shoes', 'clothing', 100),
    (10, 'Hat', 'clothing', 20)

In [0]:
Update dev.demo.products
set price = price * 2
where category = 'clothing';

In [0]:
update dev.demo.products
set price = price * 2
where category = 'electronics';
select * from dev.demo.products

In [0]:
DESCRIBE HISTORY dev.demo.products

#### 4. Use SELECT ... VERSION AS OF 

In [0]:
SELECT * from dev.demo.products VERSION AS OF 1

### Intermediate Tasks 

#### 5. Insert a row with an extra column

In [0]:
%python
products_df = spark.sql("SELECT * FROM dev.demo.products")
products_df.printSchema()

In [0]:
%python
new_product = spark.createDataFrame(
    [("11", "ipad", "electronics", "700", "0.10")],
    ["product_id", "name", "category", "price", "discount"]
)
new_product.printSchema()

In [0]:
%python
new_product.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dev.demo.products") 
    #[DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table. SQLSTATE: 42KDG


In [0]:
%python
new_product.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("dev.demo.products")

In [0]:
select * from dev.demo.products

#### 6. Use time travel

In [0]:
describe history dev.demo.products

In [0]:
SELECT *
FROM dev.demo.products VERSION AS OF 3;

In [0]:
restore table dev.demo.products to version as of 3

In [0]:
select * from dev.demo.products

#### 7. Why ACID transactions matter when multiple pipelines write to same table concurrently

When multiple pipelines write to the same table concurrently, there is always a chance of data getting lost or corrupted in between a transaction.

Think of it like a bank transaction from Bank A to Bank B:

* If an amount is being debited from Bank A, then it must be credited to Bank B. Both should happen together, otherwise nothing happens at all—keeping **atomicity**.
* If x amount is debited, then only x amount should be credited—ensuring **consistency**.
* Multiple transactions running at the same time should not disturb or interfere with each other—keeping **isolation**.
* Once the transaction is successfully completed, the changes are permanently saved and will not be lost even if there is a system failure—keeping **durability**.

### Advanced Tasks 


#### 8. Lakehouse Pipeline Design Note: Replacing Cyntexa's Nightly Batch Load

**How We Transition the Pipeline**
Instead of waiting for a heavy end-of-day dump, we can move Cyntexa to an incremental medallion lakehouse:

* **Bronze:** Ingest raw files or CDC streams continuously in micro-batches (using Auto Loader or Structured Streaming).
* **Silver & Gold:** Clean, deduplicate, and merge updates (MERGE INTO) on the fly into enriched tables and reporting-ready datasets. Data lands in minutes instead of the next morning.


**Where This Cuts Operational Risk**

**ACID Transactions:**
* If a job crashes midway, the transaction simply rolls back. You don't have to spend hours cleaning up staging tables or half-written rows.
* Analysts can query the data while pipelines are actively writing. They always see a consistent snapshot, without table locks or half-loaded data.


**Time Travel:**
* If a bad deploy or corrupted batch slips in, you can instantly roll back (RESTORE TABLE ... TO VERSION AS OF) instead of rerunning an entire nightly backfill or pulling from backups.
* You can query the table exactly as it looked at any previous point in time to trace errors or verify historical reports.

#### 9. Simulating Concurrent Writers & Analyzing the Transaction Log

![image_1787294863350.png](./image_1787294863350.png "image_1787294863350.png")


In the write_conflict job run, both writer1_query and writer2_query ran at the exact same time against the same target table.

 Both jobs read the same initial snapshot, but Writer 2 finished and committed to the Delta transaction log first.

When Writer 1 attempted to commit, Delta detected that its underlying files were already modified by Writer 2, rejecting the write with a concurrency exception to prevent data corruption.

In [0]:
describe history dev.demo.products



#### 10. Memo: Lakehouse vs. Traditional Data Warehouse

##### Advantages as Analysts

**Direct Access to Fresher, Multi-Modal Data:**  With traditional warehouses, you often have to wait for overnight updates and can only work with neat, tidy tables. A lakehouse pulls in live streams and raw logs right as they happen. You get near-instant access to everything from clean tables to messy JSON data without waiting on an engineering ticket.

**Built-in Time Travel & Reproducibility:** Delta Lake tracks every change automatically. If you need to see exactly what the data looked like last Tuesday or recreate an old report for an audit, you just query that specific timestamp. No need to ask IT to restore bulky backups or take manual snapshots.

**Unified SQL & Machine Learning Workspace:** In older setups, data science teams have to export copies of your tables to build their models, creating security headaches and stale data. In a lakehouse, everyone works off the exact same single source of truth. You can stick to plain SQL while your data science peers use Python or Spark on the exact same dataset.

##### The Tradeoff to Watch

**Higher Learning Curve & Tooling Complexity:** Traditional warehouses are plug-and-play SQL engines where things just work under the hood. Lakehouses ask analysts to learn a bit more about what's happening behind the scenes like how files are stored, how cluster sizes affect query speed, and how storage and compute work separately. It can feel like extra operational work if the team is used to simple relational databases.